In [1]:
# aqi_final_training_and_evaluation


from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import cupy as cp

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor


# -------------------- CONFIGURATION --------------------

INPUT_CSV = Path("../Datasets/Final_Dataset.csv")
OUTPUT_DIR = Path("../Outputs/XGBoost")
MODEL_DIR = OUTPUT_DIR / "trained_models"

TARGETS = ["AQI_future_1", "AQI_future_24"]
WEATHER_COLUMNS = [
    "temperature_2m (°C)",
    "relative_humidity_2m (%)",
    "dew_point_2m (°C)",
    "precipitation (mm)",
    "surface_pressure (hPa)",
    "wind_speed_10m (km/h)",
    "wind_direction_10m (°)",
    "shortwave_radiation (W/m²)",
    "cloud_cover (%)",
]

RANDOM_SEED = 42
DEVICE = "cuda"  # Change to "cpu" if CUDA is unavailable.

# Load parameters selected by the standalone tuning script.
PARAMS_FILE = OUTPUT_DIR / "best_hyperparameters.json"
if not PARAMS_FILE.exists():
    raise FileNotFoundError(
        f"Missing tuned parameters: {PARAMS_FILE.resolve()}\n"
        "Run aqi_hyperparameter_tuning_only.py first."
    )

with open(PARAMS_FILE, "r", encoding="utf-8") as f:
    PARAMS = json.load(f)

for target in TARGETS:
    if target not in PARAMS or set(PARAMS[target]) != {"A", "B"}:
        raise ValueError(
            f"Tuned parameter file must contain Model A and B for {target}."
        )


# Never use these as model inputs.
EXCLUDE = {
    "Day", "Month", "Year", "Time", "Day_of_Week", "Season",
    "Station_ID", "Station_Class", "Station_Split",
    "_Timestamp", "_year", "_target_time",
    *TARGETS,
}


# -------------------- DATA PREPARATION --------------------

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns.")


# Build timestamp from the known date/time columns.
date_text = (
    df["Year"].astype("Int64").astype(str) + "-"
    + df["Month"].astype("Int64").astype(str) + "-"
    + df["Day"].astype("Int64").astype(str) + " "
    + df["Time"].astype(str).str.strip()
)

df["_Timestamp"] = pd.to_datetime(
    date_text,
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

df.dropna(subset=["_Timestamp"], inplace=True)
df["_year"] = df["_Timestamp"].dt.year

# Derive Hour from the already-parsed timestamp.
df["Hour"] = df["_Timestamp"].dt.hour


# One-hot encode calendar categories.
df = pd.get_dummies(
    df,
    columns=["Day_of_Week", "Season"],
    dtype=np.uint8
)

# Convert targets and geographic features to numeric.
numeric_required = [
    "latitude", "longitude", "elevation", *TARGETS
]

for col in numeric_required:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Use the station classifications already in the fixed dataset.
df["Station_ID"] = df["Station_ID"].astype(str).str.strip()
df["Station_Class"] = (
    df["Station_Class"].astype(str).str.strip().str.lower()
)

if not set(df["Station_Class"].dropna().unique()).issubset({"seen", "unseen"}):
    raise ValueError("Station_Class must contain only 'Seen' and 'Unseen'.")

# Each station must belong to exactly one class.
class_counts = df.groupby("Station_ID")["Station_Class"].nunique()

if (class_counts > 1).any():
    bad_ids = class_counts[class_counts > 1].index.tolist()
    raise ValueError(f"Stations have inconsistent classifications: {bad_ids[:10]}")

seen_ids = set(df.loc[df["Station_Class"] == "seen", "Station_ID"])
unseen_ids = set(df.loc[df["Station_Class"] == "unseen", "Station_ID"])

if not seen_ids or not unseen_ids:
    raise ValueError("Both seen and unseen station groups must be present.")

df["Station_Split"] = df["Station_Class"]

df[
    ["Station_ID", "Station_Class", "Station_Split",
     "latitude", "longitude", "elevation"]
].drop_duplicates("Station_ID").to_csv(
    OUTPUT_DIR / "station_split.csv", index=False
)

print(f"Seen stations: {len(seen_ids)}")
print(f"Unseen stations: {len(unseen_ids)}")


# -------------------- FEATURE SETS --------------------

# Include numeric columns only; target and helper columns are excluded.
features = [
    col for col in df.select_dtypes(include=np.number).columns
    if col not in EXCLUDE
    and not col.startswith("AQI_future_")
    and not col.lower().startswith(("target", "label", "future_"))
]

missing_weather = [c for c in WEATHER_COLUMNS if c not in df.columns]
if missing_weather:
    raise KeyError(f"Missing expected weather columns: {missing_weather}")

model_features = {
    "A": [c for c in features if c not in WEATHER_COLUMNS],
    "B": features,
}

tuning_features_file = OUTPUT_DIR / "tuning_feature_columns.json"

if not tuning_features_file.exists():
    raise FileNotFoundError(
        "Missing tuning_feature_columns.json. "
        "Run the tuning script first."
    )

with open(tuning_features_file, encoding="utf-8") as f:
    tuning_features = json.load(f)

for name in ["A", "B"]:
    if tuning_features[name] != model_features[name]:
        raise ValueError(
            f"Feature names/order mismatch for Model {name}."
        )

print("PASS: tuning and final feature names/order match.")

for name, cols in model_features.items():
    print(f"Model {name}: {len(cols)} features")

with open(OUTPUT_DIR / "feature_columns.json", "w") as f:
    json.dump(model_features, f, indent=2)


# -------------------- TRAIN, VALIDATE, RETRAIN, AND TEST --------------------

def evaluate(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "N": len(y_true),
    }




def predict_gpu_batched(model, X, batch_size=65536):
    """
    GPU-side batched prediction to avoid CPU/GPU device-mismatch fallback.
    X must have the exact feature order used during training.
    """
    booster = model.get_booster()
    predictions = []

    for start in range(0, len(X), batch_size):
        stop = min(start + batch_size, len(X))
        cpu_batch = X.iloc[start:stop].to_numpy(
            dtype=np.float32, copy=True
        )
        gpu_batch = cp.asarray(cpu_batch)

        batch_predictions = booster.inplace_predict(
            gpu_batch,
            validate_features=False,
        )
        predictions.append(cp.asnumpy(batch_predictions))

        del gpu_batch, cpu_batch, batch_predictions

    return np.concatenate(predictions) if predictions else np.empty(0)


def make_model(params):
    return XGBRegressor(
        **params,
        objective="reg:squarederror",
        eval_metric="mae",
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method="hist",
        device=DEVICE,
    )


metrics = []

# Remove old outputs to prevent mixing results from different runs.
for target in TARGETS:
    prediction_file = OUTPUT_DIR / f"predictions_{target}.csv"
    if prediction_file.exists():
        prediction_file.unlink()

for target in TARGETS:
    horizon = 1 if target == "AQI_future_1" else 24

    # Keep only rows with valid target values.
    data = df.dropna(subset=[target]).copy()

    # Calculate when each forecast target occurs.
    data["_target_time"] = (
        data["_Timestamp"] + pd.to_timedelta(horizon, unit="h")
    )

    # -------------------- INITIAL TRAINING: 2020-2023 --------------------

    train = data[
        data["_year"].between(2020, 2023)
        & (data["_target_time"] < "2024-01-01")
        & data["Station_ID"].isin(seen_ids)
    ]

    # -------------------- VALIDATION: 2024 --------------------

    validation = data[
        (data["_year"] == 2024)
        & (data["_target_time"] < "2025-01-01")
        & data["Station_ID"].isin(seen_ids)
    ]

    # -------------------- FINAL TRAINING: 2020-2024 --------------------

    # Combine training and validation periods after hyperparameters
    # have been finalized. Only seen stations are used for training.
    final_train = data[
        data["_year"].between(2020, 2024)
        & (data["_target_time"] < "2025-01-01")
        & data["Station_ID"].isin(seen_ids)
    ]

    # -------------------- FINAL TEST SETS: 2025 --------------------

    test_seen = data[
        (data["_year"] == 2025)
        & (data["_target_time"] < "2026-01-01")
        & data["Station_ID"].isin(seen_ids)
    ]

    test_unseen = data[
        (data["_year"] == 2025)
        & (data["_target_time"] < "2026-01-01")
        & data["Station_ID"].isin(unseen_ids)
    ]

    if any(part.empty for part in [
        train, validation, final_train, test_seen, test_unseen
    ]):
        raise ValueError(
            f"{target}: an essential split is empty. "
            f"Train={len(train)}, Validation={len(validation)}, "
            f"Final train={len(final_train)}, "
            f"Seen test={len(test_seen)}, "
            f"Unseen test={len(test_unseen)}"
        )

    print(f"\n{'=' * 65}")
    print(f"Target: {target}")
    print(f"Initial training (2020-2023): {len(train):,}")
    print(f"Validation (2024):            {len(validation):,}")
    print(f"Final training (2020-2024):   {len(final_train):,}")
    print(f"Seen-station test (2025):     {len(test_seen):,}")
    print(f"Unseen-station test (2025):   {len(test_unseen):,}")

    for model_name, cols in model_features.items():

        # ============================================================
        # PHASE 1: Diagnostic fit on all seen stations (not the tuning validation split)
        # ============================================================

        validation_model = make_model(PARAMS[target][model_name])

        X_train = train[cols].replace([np.inf, -np.inf], np.nan)
        X_val = validation[cols].replace([np.inf, -np.inf], np.nan)

        validation_model.fit(
            X_train,
            train[target],
            verbose=False,
        )

        val_predictions = predict_gpu_batched(validation_model, X_val)

        metrics.append({
            "Target": target,
            "Model": model_name,
            "Evaluation": "validation_seen_2024",
            "Training_Period": "2020-2023",
            **evaluate(validation[target], val_predictions),
        })

        print(
            f"{model_name} validation MAE: "
            f"{metrics[-1]['MAE']:.4f}"
        )

        # ============================================================
        # PHASE 2: Retrain on 2020-2024 with fixed hyperparameters
        # ============================================================

        final_model = make_model(PARAMS[target][model_name])

        X_final_train = final_train[cols].replace(
            [np.inf, -np.inf], np.nan
        )

        final_model.fit(
            X_final_train,
            final_train[target],
            verbose=False,
        )

        # Save the final model trained on 2020-2024.
        joblib.dump(
            final_model,
            MODEL_DIR / f"{model_name}_{target}.joblib",
        )

        # ============================================================
        # PHASE 3: Evaluate final model on 2025 test sets
        # ============================================================

        evaluation_sets = [
            ("seen_stations_2025", test_seen),
            ("unseen_stations_2025", test_unseen),
        ]

        for split_name, test_df in evaluation_sets:
            X_test = test_df[cols].replace(
                [np.inf, -np.inf], np.nan
            )

            predictions = predict_gpu_batched(final_model, X_test)

            result = {
                "Target": target,
                "Model": model_name,
                "Evaluation": split_name,
                "Training_Period": "2020-2024",
                **evaluate(test_df[target], predictions),
            }
            metrics.append(result)

            print(
                f"{model_name} {split_name} | "
                f"MAE: {result['MAE']:.4f} | "
                f"RMSE: {result['RMSE']:.4f} | "
                f"R2: {result['R2']:.4f}"
            )

            # Save final 2025 test predictions.
            output = test_df[
                [
                    "_Timestamp",
                    "Station_ID",
                    "Station_Split",
                    "latitude",
                    "longitude",
                    "elevation",
                    target,
                ]
            ].copy()

            output["Model"] = model_name
            output["Training_Period"] = "2020-2024"
            output["Prediction"] = predictions
            output["Residual"] = output[target] - predictions

            output.to_csv(
                OUTPUT_DIR / f"predictions_{target}.csv",
                mode="a",
                header=not (
                    OUTPUT_DIR / f"predictions_{target}.csv"
                ).exists(),
                index=False,
            )

# Save all validation and final test metrics.
metrics_df = pd.DataFrame(metrics)
metrics_df.to_csv(OUTPUT_DIR / "metrics.csv", index=False)

print("\nFinal metrics:")
print(metrics_df.to_string(index=False))
print(f"\nFinal models and results saved to: {OUTPUT_DIR.resolve()}")


Loaded 1,780,876 rows and 134 columns.


C:\Users\tanki\AppData\Local\Temp\ipykernel_25916\1175219408.py:81: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_Timestamp"] = pd.to_datetime(
C:\Users\tanki\AppData\Local\Temp\ipykernel_25916\1175219408.py:88: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["_year"] = df["_Timestamp"].dt.year
C:\Users\tanki\AppData\Local\Temp\ipykernel_25916\1175219408.py:91: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining 

Seen stations: 33
Unseen stations: 8
PASS: tuning and final feature names/order match.
Model A: 127 features
Model B: 136 features

Target: AQI_future_1
Initial training (2020-2023): 895,814
Validation (2024):            276,853
Final training (2020-2024):   1,172,667
Seen-station test (2025):     252,826
Unseen-station test (2025):   66,329
A validation MAE: 1.2825
A seen_stations_2025 | MAE: 1.2724 | RMSE: 2.9365 | R2: 0.9989
A unseen_stations_2025 | MAE: 1.1537 | RMSE: 2.3168 | R2: 0.9995
B validation MAE: 1.2877
B seen_stations_2025 | MAE: 1.2801 | RMSE: 2.9478 | R2: 0.9988
B unseen_stations_2025 | MAE: 1.1588 | RMSE: 2.3224 | R2: 0.9995

Target: AQI_future_24
Initial training (2020-2023): 895,814
Validation (2024):            276,853
Final training (2020-2024):   1,172,667
Seen-station test (2025):     252,826
Unseen-station test (2025):   66,329
A validation MAE: 24.5159
A seen_stations_2025 | MAE: 22.3821 | RMSE: 35.3655 | R2: 0.8350
A unseen_stations_2025 | MAE: 21.4507 | RMSE:

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path("./temporal_spatial_results_NEW")
TARGETS = ["AQI_future_1", "AQI_future_24"]

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ============================================================
# 1. LOAD PREDICTIONS
# ============================================================

frames = []

for target in TARGETS:
    path = OUTPUT_DIR / f"predictions_{target}.csv"

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path)

    required = {
        "_Timestamp",
        "Station_ID",
        "Station_Split",
        "Model",
        target,
        "Prediction",
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{path.name} is missing columns: {sorted(missing)}"
        )

    df["Target"] = target

    df["Station_Split"] = (
        df["Station_Split"].astype(str).str.strip().str.lower()
    )

    if not df["Station_Split"].isin(["seen", "unseen"]).all():
        raise ValueError(
            f"Unexpected station groups in {path.name}"
        )

    df[target] = pd.to_numeric(df[target], errors="coerce")
    df["Prediction"] = pd.to_numeric(
        df["Prediction"], errors="coerce"
    )

    frames.append(df)

predictions = pd.concat(frames, ignore_index=True)

# Prevent duplicate records from silently inflating the metrics.
duplicate_keys = [
    "Target", "Model", "Station_ID", "_Timestamp"
]

if predictions.duplicated(duplicate_keys).any():
    raise ValueError(
        "Duplicate prediction records found. "
        "Resolve them before calculating metrics."
    )


# ============================================================
# 2. OVERALL METRICS
# ============================================================

overall_rows = []

for (target, model, split), group in predictions.groupby(
    ["Target", "Model", "Station_Split"]
):
    actual = group[target].to_numpy(dtype=float)
    predicted = group["Prediction"].to_numpy(dtype=float)

    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual = actual[valid]
    predicted = predicted[valid]

    if len(actual) == 0:
        continue

    residual = actual - predicted
    total_variance = np.sum((actual - actual.mean()) ** 2)

    overall_rows.append({
        "Target": target,
        "Model": model,
        "Station_Group": split,
        "N": len(actual),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": (
            1 - np.sum(residual ** 2) / total_variance
            if total_variance > 0 else np.nan
        ),
        "Bias": residual.mean(),
    })

overall = pd.DataFrame(overall_rows)

overall.to_csv(
    OUTPUT_DIR / "metrics_2025.csv",
    index=False,
)


# ============================================================
# 3. PER-STATION MAE AND RMSE
# ============================================================

station_rows = []

for (target, model, split, station), group in predictions.groupby(
    ["Target", "Model", "Station_Split", "Station_ID"]
):
    actual = group[target].to_numpy(dtype=float)
    predicted = group["Prediction"].to_numpy(dtype=float)

    valid = np.isfinite(actual) & np.isfinite(predicted)
    actual = actual[valid]
    predicted = predicted[valid]

    if len(actual) == 0:
        continue

    residual = actual - predicted

    station_rows.append({
        "Target": target,
        "Model": model,
        "Station_Group": split,
        "Station_ID": station,
        "N": len(actual),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "Bias": residual.mean(),
    })

station_metrics = pd.DataFrame(station_rows)

station_metrics.to_csv(
    OUTPUT_DIR / "per_station_metrics_2025.csv",
    index=False,
)


# ============================================================
# 4. STATION-LEVEL SUMMARY
# ============================================================

# Each station receives equal weight in this summary.
# This shows typical performance across stations, not rows.

station_summary = (
    station_metrics
    .groupby(["Target", "Model", "Station_Group"])
    .agg(
        Num_Stations=("Station_ID", "nunique"),
        Mean_MAE=("MAE", "mean"),
        Median_MAE=("MAE", "median"),
        Worst_MAE=("MAE", "max"),
    )
    .reset_index()
)

station_summary.to_csv(
    OUTPUT_DIR / "seen_vs_unseen_error_summary_2025.csv",
    index=False,
)


# ============================================================
# 5. PRINT COMPACT RESULTS
# ============================================================

print("\nOVERALL PERFORMANCE")
print(
    overall.round(3).to_string(index=False)
)

print("\nPERFORMANCE ACROSS STATIONS")
print(
    station_summary.round(3).to_string(index=False)
)

print("\nWORST 5 STATIONS BY MAE")

worst_stations = (
    station_metrics
    .sort_values("MAE", ascending=False)
    .groupby(
        ["Target", "Model", "Station_Group"],
        group_keys=False,
    )
    .head(5)
)

print(worst_stations.round(3).to_string(index=False))

print("\nSaved 3 analysis files to:", OUTPUT_DIR.resolve())


OVERALL PERFORMANCE
       Target Model Station_Group      N    MAE   RMSE    R2   Bias
 AQI_future_1     A          seen 252826  1.272  2.937 0.999  0.052
 AQI_future_1     A        unseen  66329  1.154  2.317 1.000  0.063
 AQI_future_1     B          seen 252826  1.280  2.948 0.999  0.035
 AQI_future_1     B        unseen  66329  1.159  2.322 1.000  0.060
AQI_future_24     A          seen 252826 22.382 35.365 0.835 -1.137
AQI_future_24     A        unseen  66329 21.451 35.059 0.893  0.511
AQI_future_24     B          seen 252826 22.161 34.783 0.840 -1.882
AQI_future_24     B        unseen  66329 21.003 34.133 0.899 -0.079

PERFORMANCE ACROSS STATIONS
       Target Model Station_Group  Num_Stations  Mean_MAE  Median_MAE  Worst_MAE
 AQI_future_1     A          seen            33     1.320       1.310      2.910
 AQI_future_1     A        unseen             8     1.138       0.837      2.669
 AQI_future_1     B          seen            33     1.328       1.323      2.902
 AQI_future_1 

In [14]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# CONFIGURATION
# ============================================================

OUTPUT_DIR = Path("./temporal_spatial_results_NEW/extra")

TARGETS = ["AQI_future_1", "AQI_future_24"]

BOOTSTRAP_REPS = 10_000
FAMILY_CONFIDENCE = 0.95
RANDOM_SEED = 42

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

rng = np.random.default_rng(RANDOM_SEED)


# ============================================================
# 1. LOAD AND VALIDATE PAIRED PREDICTIONS
# ============================================================

paired_frames = []

for target in TARGETS:
    path = OUTPUT_DIR / f"predictions_{target}.csv"

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    df = pd.read_csv(path)

    required = {
        "_Timestamp",
        "Station_ID",
        "Station_Split",
        "Model",
        target,
        "Prediction",
    }

    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{path.name} is missing columns: {sorted(missing)}"
        )

    df["Model"] = (
        df["Model"].astype(str).str.strip().str.upper()
    )

    if not df["Model"].isin(["A", "B"]).all():
        raise ValueError(
            f"{path.name} must contain Model A and Model B only."
        )

    df["Station_Split"] = (
        df["Station_Split"].astype(str).str.strip().str.lower()
    )

    if not df["Station_Split"].isin(["seen", "unseen"]).all():
        raise ValueError(
            f"Unexpected station groups in {path.name}."
        )

    df["_Timestamp"] = pd.to_datetime(
        df["_Timestamp"], errors="coerce"
    )

    df[target] = pd.to_numeric(df[target], errors="coerce")
    df["Prediction"] = pd.to_numeric(
        df["Prediction"], errors="coerce"
    )

    if df[
        ["Station_ID", "Station_Split", "_Timestamp"]
    ].isna().any().any():
        raise ValueError(
            f"{path.name} contains missing station IDs, "
            "station groups, or invalid timestamps."
        )

    keys = ["Station_ID", "Station_Split", "_Timestamp"]

    # Each model can predict each observation only once.
    if df.duplicated(keys + ["Model"]).any():
        raise ValueError(
            f"Duplicate station/timestamp/model records in {path.name}."
        )

    # Verify that both models use the same actual AQI.
    actual_counts = (
        df.groupby(keys)[target].nunique(dropna=False)
    )

    if (actual_counts > 1).any():
        raise ValueError(
            f"Actual AQI differs between models in {path.name}."
        )

    actual = (
        df.groupby(keys)[target]
        .first()
        .rename("Actual")
    )

    predictions_wide = df.pivot(
        index=keys,
        columns="Model",
        values="Prediction",
    )

    if not {"A", "B"}.issubset(predictions_wide.columns):
        raise ValueError(
            f"Both Model A and Model B are required in {path.name}."
        )

    if predictions_wide[["A", "B"]].isna().any().any():
        raise ValueError(
            f"Missing paired predictions in {path.name}. "
            "Resolve them before running the test."
        )

    paired = actual.to_frame().join(
        predictions_wide[["A", "B"]]
    ).reset_index()

    paired = paired.rename(columns={
        "A": "Prediction_A",
        "B": "Prediction_B",
    })

    paired["Target"] = target

    valid = (
        np.isfinite(paired["Actual"])
        & np.isfinite(paired["Prediction_A"])
        & np.isfinite(paired["Prediction_B"])
    )

    removed = int((~valid).sum())

    if removed:
        print(
            f"{target}: excluding {removed:,} rows "
            "with invalid actuals or predictions."
        )

    paired = paired.loc[valid].copy()

    paired_frames.append(paired)

paired_data = pd.concat(paired_frames, ignore_index=True)

if paired_data.empty:
    raise ValueError("No valid paired predictions remain.")

# A station must belong to only one split.
split_counts = (
    paired_data.groupby("Station_ID")["Station_Split"].nunique()
)

if (split_counts > 1).any():
    raise ValueError(
        "At least one station belongs to multiple station groups."
    )

print(f"Valid paired observations: {len(paired_data):,}")

print(
    "Stations by group:",
    paired_data.groupby("Station_Split")["Station_ID"]
    .nunique().to_dict(),
)


# ============================================================
# 2. CALCULATE METRICS FOR EACH STATION
# ============================================================

station_rows = []

for (target, split, station_id), group in paired_data.groupby(
    ["Target", "Station_Split", "Station_ID"]
):
    actual = group["Actual"].to_numpy(dtype=float)
    pred_a = group["Prediction_A"].to_numpy(dtype=float)
    pred_b = group["Prediction_B"].to_numpy(dtype=float)

    error_a = actual - pred_a
    error_b = actual - pred_b

    mae_a = np.mean(np.abs(error_a))
    mae_b = np.mean(np.abs(error_b))

    rmse_a = np.sqrt(np.mean(error_a ** 2))
    rmse_b = np.sqrt(np.mean(error_b ** 2))

    station_rows.append({
        "Target": target,
        "Station_Group": split,
        "Station_ID": station_id,
        "N": len(group),

        "MAE_A": mae_a,
        "MAE_B": mae_b,
        "Delta_MAE_B_minus_A": mae_b - mae_a,

        "RMSE_A": rmse_a,
        "RMSE_B": rmse_b,
    })

station_metrics = pd.DataFrame(station_rows)

station_metrics.to_csv(
    OUTPUT_DIR / "ab_test_station_comparison_2025.csv",
    index=False,
)


# ============================================================
# 3. BOOTSTRAP HELPER
# ============================================================

def stratified_station_bootstrap(
    station_group,
    repetitions,
    random_generator,
):
    """
    Bootstrap the mean station-level MAE difference.

    Resample stations with replacement separately within the
    seen and unseen groups, preserving each group's station count.

    Every station receives equal weight in the final estimate.
    """

    differences_by_group = [
        group["Delta_MAE_B_minus_A"].to_numpy(dtype=float)
        for _, group in station_group.groupby("Station_Group")
    ]

    if any(len(values) == 0 for values in differences_by_group):
        raise ValueError("An empty station group was encountered.")

    bootstrap_deltas = np.empty(repetitions)

    for i in range(repetitions):
        sampled_groups = []

        for differences in differences_by_group:
            sampled = random_generator.choice(
                differences,
                size=len(differences),
                replace=True,
            )
            sampled_groups.extend(sampled)

        bootstrap_deltas[i] = np.mean(sampled_groups)

    return bootstrap_deltas


# ============================================================
# 4. TWO PRIMARY EXPERIMENTS
# ============================================================

summary_rows = []

# Two primary comparisons:
# 1. AQI_future_1
# 2. AQI_future_24
#
# Bonferroni correction controls family-wise error at 5%.
# Each primary interval therefore has 97.5% nominal coverage.

number_of_primary_tests = len(TARGETS)
alpha = 1 - FAMILY_CONFIDENCE
adjusted_alpha = alpha / number_of_primary_tests

lower_quantile = adjusted_alpha / 2
upper_quantile = 1 - adjusted_alpha / 2


for target in TARGETS:

    target_stations = station_metrics[
        station_metrics["Target"] == target
    ].copy()

    if target_stations.empty:
        raise ValueError(f"No station results found for {target}.")

    # --------------------------------------------------------
    # PRIMARY COMPARISON: ALL ELIGIBLE STATIONS
    # --------------------------------------------------------

    station_deltas = target_stations[
        "Delta_MAE_B_minus_A"
    ].to_numpy(dtype=float)

    n_stations = len(target_stations)

    # Station-weighted MAE:
    # calculate each station's MAE first, then average stations.
    macro_mae_a = target_stations["MAE_A"].mean()
    macro_mae_b = target_stations["MAE_B"].mean()

    macro_delta = macro_mae_b - macro_mae_a

    bootstrap_deltas = stratified_station_bootstrap(
        target_stations,
        BOOTSTRAP_REPS,
        rng,
    )

    ci_low, ci_high = np.quantile(
        bootstrap_deltas,
        [lower_quantile, upper_quantile],
    )

    relative_improvement = (
        (macro_mae_a - macro_mae_b) / macro_mae_a * 100
        if macro_mae_a != 0 else np.nan
    )

    if ci_high < 0:
        conclusion = "Weather features significantly improve MAE"
    elif ci_low > 0:
        conclusion = "Weather features significantly worsen MAE"
    else:
        conclusion = "No clear difference detected"

    summary_rows.append({
        "Target": target,
        "Analysis": "PRIMARY_ALL_STATIONS",

        "N_Stations": n_stations,
        "N_Observations": int(target_stations["N"].sum()),

        "Station_Weighted_MAE_A": macro_mae_a,
        "Station_Weighted_MAE_B": macro_mae_b,
        "Delta_MAE_B_minus_A": macro_delta,
        "Relative_MAE_Improvement_Pct": relative_improvement,

        "CI_Lower": ci_low,
        "CI_Upper": ci_high,
        "CI_Confidence_Level": 1 - adjusted_alpha,

        "Conclusion": conclusion,
    })

    # --------------------------------------------------------
    # SECONDARY COMPARISONS: SEEN AND UNSEEN STATIONS
    # --------------------------------------------------------
    # These are descriptive subgroup analyses, not additional
    # primary significance tests.

    for split in ["seen", "unseen"]:

        subgroup = target_stations[
            target_stations["Station_Group"] == split
        ]

        if subgroup.empty:
            continue

        mae_a = subgroup["MAE_A"].mean()
        mae_b = subgroup["MAE_B"].mean()

        delta = mae_b - mae_a

        relative = (
            (mae_a - mae_b) / mae_a * 100
            if mae_a != 0 else np.nan
        )

        summary_rows.append({
            "Target": target,
            "Analysis": f"SECONDARY_{split.upper()}",

            "N_Stations": subgroup["Station_ID"].nunique(),
            "N_Observations": int(subgroup["N"].sum()),

            "Station_Weighted_MAE_A": mae_a,
            "Station_Weighted_MAE_B": mae_b,
            "Delta_MAE_B_minus_A": delta,
            "Relative_MAE_Improvement_Pct": relative,

            # No inferential interval for these secondary results.
            "CI_Lower": np.nan,
            "CI_Upper": np.nan,
            "CI_Confidence_Level": np.nan,

            "Conclusion": "Descriptive subgroup result only",
        })


# ============================================================
# 5. SAVE AND DISPLAY RESULTS
# ============================================================

summary = pd.DataFrame(summary_rows)

summary.to_csv(
    OUTPUT_DIR / "ab_test_summary_2025.csv",
    index=False,
)

print("\n" + "=" * 105)
print("WEATHER FEATURE A/B TEST — 2025 TEST SET")
print("=" * 105)

display_columns = [
    "Target",
    "Analysis",
    "N_Stations",
    "Station_Weighted_MAE_A",
    "Station_Weighted_MAE_B",
    "Delta_MAE_B_minus_A",
    "Relative_MAE_Improvement_Pct",
    "CI_Lower",
    "CI_Upper",
    "Conclusion",
]

print(
    summary[display_columns]
    .round(4)
    .to_string(index=False)
)



print("\nSaved files:")
print(OUTPUT_DIR / "ab_test_summary_2025.csv")
print(OUTPUT_DIR / "ab_test_station_comparison_2025.csv")

Valid paired observations: 638,310
Stations by group: {'seen': 33, 'unseen': 8}

WEATHER FEATURE A/B TEST — 2025 TEST SET
       Target             Analysis  N_Stations  Station_Weighted_MAE_A  Station_Weighted_MAE_B  Delta_MAE_B_minus_A  Relative_MAE_Improvement_Pct  CI_Lower  CI_Upper                                 Conclusion
 AQI_future_1 PRIMARY_ALL_STATIONS          41                  1.2842                  1.2915               0.0073                       -0.5672    0.0026    0.0120  Weather features significantly worsen MAE
 AQI_future_1       SECONDARY_SEEN          33                  1.3197                  1.3276               0.0079                       -0.5965       NaN       NaN           Descriptive subgroup result only
 AQI_future_1     SECONDARY_UNSEEN           8                  1.1377                  1.1426               0.0049                       -0.4272       NaN       NaN           Descriptive subgroup result only
AQI_future_24 PRIMARY_ALL_STATIONS        